# Project 31 — ChemGuard AI: Laboratory Quality Control, Confidence-Gated Classification & Drift Intelligence

## Real chemical-analysis data + classification + uncertainty routing + anomaly detection + drift monitoring

**Goal:** build an end-to-end quality-intelligence system that does more than predict a class. ChemGuard AI must also decide **when to trust the model, when to route a sample for manual review, and when incoming chemistry has drifted away from the training population**.

### Why this is portfolio-grade
- Real **UCI Wine** chemical-analysis measurements bundled with scikit-learn
- Competing classification models with 5-fold cross-validation
- Locked hold-out test set
- Confidence-based human-review routing
- Calibration error measurement
- Permutation importance for explainability
- PCA visual quality map
- Isolation Forest anomaly screening
- Noise robustness testing
- Population Stability Index (PSI) drift monitoring
- Production-style prediction guardrails and executive scorecard

> The dataset contains laboratory measurements from 178 chemically analysed samples across three cultivars. The project uses the data as a generic laboratory-quality classification benchmark.


## 1. Imports and reproducibility
The notebook is intentionally self-contained and uses only packages normally available in a standard Colab/scikit-learn environment.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, IsolationForest
from sklearn.metrics import (
    accuracy_score, f1_score, log_loss, confusion_matrix,
    classification_report
)
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA

SEED = 42
np.random.seed(SEED)

print("ChemGuard AI environment ready ✅")


ChemGuard AI environment ready ✅


## 2. Load the real UCI chemical-analysis dataset


In [2]:
wine = load_wine(as_frame=True)

X = wine.data.copy()
y = wine.target.copy()
target_names = list(wine.target_names)

print("Dataset shape:", X.shape)
print("Classes:", target_names)
print("Features:", X.columns.tolist())
display(X.head())


Dataset shape: (178, 13)
Classes: ['class_0', 'class_1', 'class_2']
Features: ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']


## 3. Data-quality audit
A production classifier should fail early when the input schema is incomplete or corrupted.


In [3]:
quality = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "missing_n": X.isna().sum(),
    "missing_pct": X.isna().mean() * 100,
    "unique_n": X.nunique()
})

print("Missing values:", int(X.isna().sum().sum()))
print("Duplicate rows:", int(X.duplicated().sum()))
print("Class counts:")
print(y.value_counts().sort_index().to_string())
display(quality)


Missing values: 0
Duplicate rows: 0
Class counts:
target
0    59
1    71
2    48


## 4. Exploratory chemistry map
The first visual layer compares class balance and feature distributions before any modelling.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

y.value_counts().sort_index().plot(kind="bar", ax=axes[0])
axes[0].set_title("Laboratory Sample Class Balance")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Samples")

X[["alcohol", "flavanoids", "color_intensity", "proline"]].boxplot(ax=axes[1], rot=30)
axes[1].set_title("Selected Chemical Feature Distributions")

plt.tight_layout()
plt.show()


## 5. Locked train/test design
The test set is held out before model comparison. Stratification preserves the three-class distribution.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

print("Training samples:", len(X_train))
print("Locked test samples:", len(X_test))
print("Train class distribution:", y_train.value_counts(normalize=True).sort_index().round(3).to_dict())
print("Test class distribution:", y_test.value_counts(normalize=True).sort_index().round(3).to_dict())


Training samples: 124
Locked test samples: 54
Train class distribution: {0: 0.331, 1: 0.403, 2: 0.266}
Test class distribution: {0: 0.333, 1: 0.389, 2: 0.278}


## 6. Competing models
We deliberately compare a linear calibrated-style baseline with two nonlinear ensemble families.


In [5]:
models = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, random_state=SEED))
    ]),
    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.06,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=SEED
    )
}

print("Candidate models:", list(models))


Candidate models: ['LogisticRegression', 'RandomForest', 'HistGradientBoosting']


## 7. Five-fold model selection
The selection criterion prioritises **macro-F1**, then lower log loss. This prevents a high headline accuracy from hiding weak minority-class performance.


In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

rows = []
for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "macro_f1": "f1_macro",
            "log_loss": "neg_log_loss"
        }
    )
    rows.append({
        "model": name,
        "cv_accuracy": scores["test_accuracy"].mean(),
        "cv_accuracy_sd": scores["test_accuracy"].std(),
        "cv_macro_f1": scores["test_macro_f1"].mean(),
        "cv_macro_f1_sd": scores["test_macro_f1"].std(),
        "cv_log_loss": -scores["test_log_loss"].mean()
    })

cv_results = pd.DataFrame(rows).sort_values(
    ["cv_macro_f1", "cv_log_loss"],
    ascending=[False, True]
)

display(cv_results.round(4))
BEST_NAME = cv_results.iloc[0]["model"]
print("Selected model:", BEST_NAME)


               model  cv_accuracy  cv_accuracy_sd  cv_macro_f1  cv_macro_f1_sd  cv_log_loss
  LogisticRegression        0.9840          0.0196       0.9834          0.0203       0.0566
        RandomForest        0.9760          0.0320       0.9760          0.0321       0.1895
HistGradientBoosting        0.9680          0.0466       0.9690          0.0449       0.0696
Selected model: LogisticRegression


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
ordered = cv_results.sort_values("cv_macro_f1")
ax.barh(ordered["model"], ordered["cv_macro_f1"])
ax.set_xlim(0.90, 1.0)
ax.set_xlabel("5-fold macro-F1")
ax.set_title("ChemGuard Model Benchmark")
plt.tight_layout()
plt.show()


## 8. Locked-test performance


In [7]:
best_model = models[BEST_NAME].fit(X_train, y_train)

pred = best_model.predict(X_test)
proba = best_model.predict_proba(X_test)

TEST_ACCURACY = accuracy_score(y_test, pred)
TEST_MACRO_F1 = f1_score(y_test, pred, average="macro")
TEST_LOG_LOSS = log_loss(y_test, proba)

print(f"Best model: {BEST_NAME}")
print(f"Test accuracy: {TEST_ACCURACY:.4f}")
print(f"Test macro-F1: {TEST_MACRO_F1:.4f}")
print(f"Test log loss: {TEST_LOG_LOSS:.4f}")
print()
print(confusion_matrix(y_test, pred))


Best model: LogisticRegression
Test accuracy: 0.9815
Test macro-F1: 0.9829
Test log loss: 0.0778

[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


In [ ]:
report = classification_report(
    y_test,
    pred,
    target_names=target_names,
    digits=3,
    output_dict=True
)
display(pd.DataFrame(report).T)


In [ ]:
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm)
ax.set_xticks(range(3), target_names, rotation=30)
ax.set_yticks(range(3), target_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("ChemGuard Locked-Test Confusion Matrix")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()


## 9. Confidence-aware human-review routing
A useful production system should be allowed to say **"review this sample"** rather than forcing a prediction.

We measure **coverage** (fraction automatically accepted) against **selective accuracy** (accuracy among accepted samples).


In [8]:
confidence = proba.max(axis=1)
correct = pred == y_test.to_numpy()

routing_rows = []
for threshold in [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]:
    auto = confidence >= threshold
    routing_rows.append({
        "confidence_threshold": threshold,
        "auto_coverage": auto.mean(),
        "manual_review_n": int((~auto).sum()),
        "selective_accuracy": correct[auto].mean() if auto.any() else np.nan
    })

routing = pd.DataFrame(routing_rows)
display(routing.round(4))


 confidence_threshold  auto_coverage  manual_review_n  selective_accuracy
                 0.50         0.9815                1                 1.0
                 0.60         0.9815                1                 1.0
                 0.70         0.9630                2                 1.0
                 0.80         0.9074                5                 1.0
                 0.90         0.8333                9                 1.0
                 0.95         0.6852               17                 1.0


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(routing["auto_coverage"], routing["selective_accuracy"], marker="o")
for _, row in routing.iterrows():
    ax.annotate(
        f"{row['confidence_threshold']:.2f}",
        (row["auto_coverage"], row["selective_accuracy"]),
        xytext=(5, 5),
        textcoords="offset points"
    )
ax.set_xlabel("Automatic coverage")
ax.set_ylabel("Accuracy of automatically accepted samples")
ax.set_title("Confidence Routing — Automation vs Reliability")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 10. Confidence calibration diagnostic
Expected Calibration Error (ECE) compares average model confidence with observed correctness across confidence bins.


In [9]:
bins = np.linspace(0, 1, 11)
ECE = 0.0
calibration_rows = []

for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (confidence > lo) & (confidence <= hi)
    if mask.any():
        bin_acc = correct[mask].mean()
        bin_conf = confidence[mask].mean()
        weight = mask.mean()
        ECE += weight * abs(bin_acc - bin_conf)
        calibration_rows.append({
            "lower": lo,
            "upper": hi,
            "n": int(mask.sum()),
            "accuracy": bin_acc,
            "mean_confidence": bin_conf
        })

calibration = pd.DataFrame(calibration_rows)
print(f"Expected Calibration Error: {ECE:.4f}")
display(calibration.round(4))


Expected Calibration Error: 0.0560


## 11. Explainability with permutation importance
Permutation importance asks how much predictive performance falls when each chemical feature is independently shuffled.


In [10]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=50,
    random_state=SEED,
    scoring="f1_macro",
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X.columns,
    "macro_f1_drop": perm.importances_mean,
    "std": perm.importances_std
}).sort_values("macro_f1_drop", ascending=False)

display(importance.head(10).round(4))


                     feature  macro_f1_drop    std
                     proline         0.0653 0.0268
                     alcohol         0.0636 0.0250
                  flavanoids         0.0385 0.0205
                         ash         0.0378 0.0220
                         hue         0.0373 0.0172
             color_intensity         0.0266 0.0164
           alcalinity_of_ash         0.0246 0.0174
od280/od315_of_diluted_wines         0.0238 0.0126
             proanthocyanins         0.0073 0.0133
                  malic_acid         0.0015 0.0046


In [ ]:
top = importance.head(10).sort_values("macro_f1_drop")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top["feature"], top["macro_f1_drop"], xerr=top["std"])
ax.set_xlabel("Macro-F1 decrease after permutation")
ax.set_title("ChemGuard Explainability — Predictive Chemical Signals")
plt.tight_layout()
plt.show()


## 12. PCA quality map
A two-dimensional principal-component map gives analysts a fast visual check for cluster overlap and unusual chemistry.


In [11]:
scaler_all = StandardScaler()
X_scaled = scaler_all.fit_transform(X)

pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

print("PC1 explained variance:", round(float(pca.explained_variance_ratio_[0]), 4))
print("PC2 explained variance:", round(float(pca.explained_variance_ratio_[1]), 4))
print("2D total explained variance:", round(float(pca.explained_variance_ratio_.sum()), 4))

fig, ax = plt.subplots(figsize=(8, 6))
for cls in sorted(y.unique()):
    mask = y.to_numpy() == cls
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.75, label=target_names[cls])

ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")
ax.set_title("ChemGuard Chemical-Space Quality Map")
ax.legend()
plt.tight_layout()
plt.show()


PC1 explained variance: 0.362
PC2 explained variance: 0.1921
2D total explained variance: 0.5541


## 13. Anomaly screening
Isolation Forest provides a separate **"is this sample unusual?"** signal. This is intentionally independent from class prediction.


In [12]:
anomaly_model = Pipeline([
    ("scaler", StandardScaler()),
    ("isolation_forest", IsolationForest(
        n_estimators=400,
        contamination=0.05,
        random_state=SEED
    ))
])

anomaly_model.fit(X_train)
is_anomaly = anomaly_model.predict(X_test) == -1

print("Flagged test anomalies:", int(is_anomaly.sum()))
print("Anomaly rate:", round(float(is_anomaly.mean()), 4))
print("Accuracy on non-flagged samples:", round(float(correct[~is_anomaly].mean()), 4))


Flagged test anomalies: 1
Anomaly rate: 0.0185
Accuracy on non-flagged samples: 0.9811


## 14. Robustness under measurement noise
Laboratory systems encounter measurement variation. We perturb test features by fractions of the training-set standard deviation and re-evaluate.


In [13]:
rng = np.random.default_rng(SEED)
feature_std = X_train.std().to_numpy()

noise_rows = []
for level in [0.00, 0.025, 0.05, 0.10, 0.20]:
    X_noise = X_test.copy()
    if level > 0:
        X_noise.iloc[:, :] = (
            X_noise.to_numpy()
            + rng.normal(0, level * feature_std, size=X_noise.shape)
        )
    p = best_model.predict(X_noise)
    pr = best_model.predict_proba(X_noise)
    noise_rows.append({
        "noise_fraction_of_train_sd": level,
        "accuracy": accuracy_score(y_test, p),
        "macro_f1": f1_score(y_test, p, average="macro"),
        "mean_confidence": pr.max(axis=1).mean()
    })

noise_results = pd.DataFrame(noise_rows)
display(noise_results.round(4))


 noise_fraction_of_train_sd  accuracy  macro_f1  mean_confidence
                      0.000    0.9815    0.9829           0.9423
                      0.025    0.9815    0.9829           0.9435
                      0.050    0.9815    0.9829           0.9429
                      0.100    0.9815    0.9829           0.9392
                      0.200    1.0000    1.0000           0.9299


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(noise_results["noise_fraction_of_train_sd"], noise_results["macro_f1"], marker="o", label="Macro-F1")
ax.plot(noise_results["noise_fraction_of_train_sd"], noise_results["mean_confidence"], marker="o", label="Mean confidence")
ax.set_xlabel("Injected measurement noise / training SD")
ax.set_title("ChemGuard Measurement-Noise Stress Test")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 15. Synthetic drift stress test
For monitoring validation, we intentionally shift the three strongest predictive features by **+0.5 training standard deviations**. This is not new real-world data; it is a transparent stress scenario designed to test whether the monitoring layer notices distribution change.


In [14]:
top_features = importance.head(3)["feature"].tolist()

X_drift = X_test.copy()
for feature in top_features:
    X_drift[feature] = X_drift[feature] + 0.5 * X_train[feature].std()

drift_pred = best_model.predict(X_drift)
drift_proba = best_model.predict_proba(X_drift)

print("Shifted features:", top_features)
print("Baseline macro-F1:", round(TEST_MACRO_F1, 4))
print("Drifted macro-F1:", round(f1_score(y_test, drift_pred, average="macro"), 4))
print("Baseline mean confidence:", round(float(confidence.mean()), 4))
print("Drifted mean confidence:", round(float(drift_proba.max(axis=1).mean()), 4))


Shifted features: ['proline', 'alcohol', 'flavanoids']
Baseline macro-F1: 0.9829
Drifted macro-F1: 0.9658
Baseline mean confidence: 0.9423
Drifted mean confidence: 0.9125


## 16. Population Stability Index monitoring
PSI is used here as a simple operational drift indicator. Thresholds should be calibrated to the organisation and data-generating process rather than treated as universal laws.


In [15]:
def population_stability_index(expected, actual, bins=10):
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0

    edges[0] -= 1e-9
    edges[-1] += 1e-9

    expected_pct = np.histogram(expected, bins=edges)[0] / len(expected)
    actual_pct = np.histogram(actual, bins=edges)[0] / len(actual)

    expected_pct = np.clip(expected_pct, 1e-6, None)
    actual_pct = np.clip(actual_pct, 1e-6, None)

    return float(np.sum(
        (actual_pct - expected_pct)
        * np.log(actual_pct / expected_pct)
    ))

psi_rows = []
for feature in X.columns:
    psi_rows.append({
        "feature": feature,
        "psi": population_stability_index(
            X_train[feature].to_numpy(),
            X_drift[feature].to_numpy()
        )
    })

psi_table = pd.DataFrame(psi_rows).sort_values("psi", ascending=False)
display(psi_table.head(10).round(4))


                     feature    psi
                  flavanoids 2.5623
                     alcohol 1.6945
                  malic_acid 1.5672
                     proline 1.3117
                         hue 0.3620
        nonflavanoid_phenols 0.3108
                   magnesium 0.2426
                         ash 0.2378
           alcalinity_of_ash 0.2061
od280/od315_of_diluted_wines 0.1920


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
plot_psi = psi_table.head(10).sort_values("psi")
ax.barh(plot_psi["feature"], plot_psi["psi"])
ax.axvline(0.25, linestyle="--", linewidth=1.5, label="Illustrative alert line")
ax.set_xlabel("Population Stability Index")
ax.set_title("ChemGuard Drift Monitor — Top Feature Shifts")
ax.legend()
plt.tight_layout()
plt.show()


## 17. Production guardrail API logic
The decision engine combines class probability and anomaly status into an operational action:
- **AUTO_ACCEPT** when confidence is high and chemistry is not anomalous
- **MANUAL_REVIEW** when confidence is insufficient
- **ANOMALY_REVIEW** when the sample is out-of-distribution


In [16]:
REQUIRED_FEATURES = X.columns.tolist()

def validate_sample(sample):
    missing = [f for f in REQUIRED_FEATURES if f not in sample]
    if missing:
        raise ValueError(f"Missing required features: {missing}")

    values = np.array([sample[f] for f in REQUIRED_FEATURES], dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("All feature values must be finite numbers.")

def chemguard_predict(sample, confidence_threshold=0.80):
    validate_sample(sample)

    row = pd.DataFrame([sample], columns=REQUIRED_FEATURES)
    probabilities = best_model.predict_proba(row)[0]
    predicted_class = int(np.argmax(probabilities))
    max_confidence = float(np.max(probabilities))
    anomaly_flag = bool(anomaly_model.predict(row)[0] == -1)

    if anomaly_flag:
        action = "ANOMALY_REVIEW"
    elif max_confidence < confidence_threshold:
        action = "MANUAL_REVIEW"
    else:
        action = "AUTO_ACCEPT"

    return {
        "predicted_class": target_names[predicted_class],
        "confidence": max_confidence,
        "anomaly_flag": anomaly_flag,
        "action": action,
        "class_probabilities": {
            target_names[i]: float(probabilities[i])
            for i in range(len(target_names))
        }
    }

print("Production decision function ready ✅")


Production decision function ready ✅


## 18. Example decision


In [ ]:
example_sample = X_test.iloc[0].to_dict()
example_result = chemguard_predict(example_sample, confidence_threshold=0.80)

print("True class:", target_names[int(y_test.iloc[0])])
print("ChemGuard decision:")
for key, value in example_result.items():
    print(f"{key}: {value}")


## 19. Executive reliability scorecard


In [17]:
auto_at_80 = routing.loc[routing["confidence_threshold"] == 0.80].iloc[0]

scorecard = pd.DataFrame([
    ["Model quality", "Locked-test accuracy", TEST_ACCURACY, "Overall held-out correctness"],
    ["Model quality", "Locked-test macro-F1", TEST_MACRO_F1, "Balanced class performance"],
    ["Probability quality", "Log loss", TEST_LOG_LOSS, "Penalises confident wrong predictions"],
    ["Calibration", "Expected Calibration Error", ECE, "Confidence vs observed correctness"],
    ["Guardrails", "Auto coverage at 0.80", auto_at_80["auto_coverage"], "Fraction eligible for automation"],
    ["Guardrails", "Selective accuracy at 0.80", auto_at_80["selective_accuracy"], "Accuracy among auto-accepted samples"],
    ["Monitoring", "Drift-stress macro-F1", f1_score(y_test, drift_pred, average="macro"), "Stress-test performance after feature shift"],
    ["Monitoring", "Drift-stress mean confidence", drift_proba.max(axis=1).mean(), "Confidence degradation under shift"]
], columns=["layer", "metric", "value", "decision_meaning"])

display(scorecard.round(4))


          layer                       metric  value                              decision_meaning
  Model quality         Locked-test accuracy 0.9815             Overall held-out correctness
  Model quality         Locked-test macro-F1 0.9829             Balanced class performance
Probability quality                     Log loss 0.0778   Penalises confident wrong predictions
     Calibration Expected Calibration Error 0.0560 Confidence vs observed correctness
      Guardrails          Auto coverage at 0.80 0.9074   Fraction eligible for automation
      Guardrails     Selective accuracy at 0.80 1.0000 Accuracy among auto-accepted samples
      Monitoring       Drift-stress macro-F1 0.9658 Stress-test performance after feature shift
      Monitoring Drift-stress mean confidence 0.9125 Confidence degradation under shift


## 20. Production architecture

```text
Laboratory / Sensor Measurements
            │
            ▼
   Schema + Quality Validation
            │
            ├──────────────► Reject malformed input
            ▼
      Classification Model
            │
            ├────► Probability / Confidence
            │
            └────► Predicted Sample Class
            │
            ▼
     Anomaly Detection Layer
            │
            ▼
 Confidence + OOD Decision Policy
      │          │          │
      ▼          ▼          ▼
AUTO_ACCEPT  MANUAL_REVIEW  ANOMALY_REVIEW
      │
      ▼
 Monitoring Store
      │
      ├────► PSI / feature drift
      ├────► confidence drift
      └────► model performance when labels arrive
```

This is the key distinction between a classroom classifier and a deployable decision system: **prediction, uncertainty, abstention and monitoring are designed together**.


## 21. Limitations and responsible use
- The dataset is small (178 samples), so this is a **portfolio benchmark**, not a validated industrial laboratory system.
- Synthetic drift is deliberately introduced to test monitoring behaviour; it is not presented as observed production drift.
- Confidence thresholds are operational choices and should be calibrated to costs, risk tolerance and downstream review capacity.
- Isolation Forest flags unusual chemistry but does not explain root cause.
- A real deployment would add versioned data contracts, authenticated inference, logging, alerting, retraining governance and external validation.


## 22. CV / interview talking point
**ChemGuard AI** demonstrates that I can take a real classification problem beyond model training into **reliability engineering**: cross-validation, locked testing, probability quality, human-review routing, anomaly screening, robustness testing and data-drift monitoring.


In [18]:
print("PROJECT 31 — CHEMGUARD AI FINAL RESULTS")
print("=" * 84)
print("Real dataset: UCI laboratory chemical-analysis benchmark")
print(f"Samples: {len(X)} | Features: {X.shape[1]} | Classes: {y.nunique()}")
print(f"Selected model: {BEST_NAME}")
print(f"Locked-test accuracy: {TEST_ACCURACY:.2%}")
print(f"Locked-test macro-F1: {TEST_MACRO_F1:.4f}")
print(f"Locked-test log loss: {TEST_LOG_LOSS:.4f}")
print(f"Expected Calibration Error: {ECE:.4f}")
print(f"Auto coverage @ 0.80 confidence: {auto_at_80['auto_coverage']:.2%}")
print(f"Selective accuracy @ 0.80 confidence: {auto_at_80['selective_accuracy']:.2%}")
print(f"Drift-stress macro-F1: {f1_score(y_test, drift_pred, average='macro'):.4f}")
print()
print("✅ Classification")
print("✅ Explainability")
print("✅ Confidence-aware abstention")
print("✅ Anomaly screening")
print("✅ Measurement-noise robustness")
print("✅ PSI drift monitoring")
print("✅ Production guardrail logic")
print("✅ Executive reliability scorecard")
print()
print("PROJECT 31 WOW FACTOR READY ✅")


PROJECT 31 — CHEMGUARD AI FINAL RESULTS
Real dataset: UCI laboratory chemical-analysis benchmark
Samples: 178 | Features: 13 | Classes: 3
Selected model: LogisticRegression
Locked-test accuracy: 98.15%
Locked-test macro-F1: 0.9829
Locked-test log loss: 0.0778
Expected Calibration Error: 0.0560
Auto coverage @ 0.80 confidence: 90.74%
Selective accuracy @ 0.80 confidence: 100.00%
Drift-stress macro-F1: 0.9658

✅ Classification
✅ Explainability
✅ Confidence-aware abstention
✅ Anomaly screening
✅ Measurement-noise robustness
✅ PSI drift monitoring
✅ Production guardrail logic
✅ Executive reliability scorecard

PROJECT 31 WOW FACTOR READY ✅


## 23. ChemGuard Mission Control — operational batch triage

This final layer converts the model into an **operations decision system**. Instead of asking only whether the classifier is accurate, it measures how many laboratory samples can be safely automated, how many require human review, whether the guardrail catches the observed error, and whether the data distribution has shifted enough to trigger monitoring attention.

The aim is the kind of reliability story a production ML, data science or AI engineering team can defend in an interview: **predict → quantify confidence → abstain when uncertain → screen anomalies → monitor drift → escalate**.


In [19]:
batch_actions = np.where(
    is_anomaly,
    "ANOMALY_REVIEW",
    np.where(confidence < 0.80, "MANUAL_REVIEW", "AUTO_ACCEPT")
)

batch_correct = pred == y_test.to_numpy()
action_counts = pd.Series(batch_actions).value_counts().reindex(
    ["AUTO_ACCEPT", "MANUAL_REVIEW", "ANOMALY_REVIEW"], fill_value=0
)
auto_mask = batch_actions == "AUTO_ACCEPT"
review_mask = ~auto_mask
drift_alert_features = int((psi_table["psi"] > 0.25).sum())

mission_control = pd.DataFrame([
    ["Automation", "Auto-accepted samples", int(action_counts["AUTO_ACCEPT"]), "High-confidence, non-anomalous samples"],
    ["Human review", "Manual-review samples", int(action_counts["MANUAL_REVIEW"]), "Low-confidence samples"],
    ["Anomaly control", "Anomaly-review samples", int(action_counts["ANOMALY_REVIEW"]), "Unusual chemistry"],
    ["Safety", "Auto-accepted accuracy", float(batch_correct[auto_mask].mean()), "Correctness of automated decisions"],
    ["Safety", "Observed errors auto-accepted", int((~batch_correct[auto_mask]).sum()), "Should remain zero in this test batch"],
    ["Monitoring", "PSI alert features", drift_alert_features, "Features above illustrative 0.25 PSI alert line"],
    ["Monitoring", "Drift-stress macro-F1", float(f1_score(y_test, drift_pred, average="macro")), "Performance under synthetic shift"],
], columns=["layer", "metric", "value", "operational_meaning"])

display(mission_control.round(4))
print("CHEMGUARD MISSION CONTROL")
print("=" * 84)
print(f"Locked test batch: {len(y_test)} samples")
print(f"AUTO_ACCEPT: {int(action_counts['AUTO_ACCEPT'])}")
print(f"MANUAL_REVIEW: {int(action_counts['MANUAL_REVIEW'])}")
print(f"ANOMALY_REVIEW: {int(action_counts['ANOMALY_REVIEW'])}")
print(f"Automated-decision accuracy: {batch_correct[auto_mask].mean():.2%}")
print(f"Observed model errors sent to review: {int((~batch_correct[review_mask]).sum())}/{int((~batch_correct).sum())}")
print(f"PSI alert features in drift stress test: {drift_alert_features}")
print("\n✅ Confidence gate protected the automated path from the observed classification error")


           layer                       metric   value                              operational_meaning
      Automation        Auto-accepted samples 48.0000          High-confidence, non-anomalous samples
    Human review        Manual-review samples  5.0000                       Low-confidence samples
  Anomaly control       Anomaly-review samples  1.0000                            Unusual chemistry
          Safety       Auto-accepted accuracy  1.0000              Correctness of automated decisions
          Safety Observed errors auto-accepted  0.0000           Should remain zero in this test batch
      Monitoring           PSI alert features  6.0000 Features above illustrative 0.25 PSI alert line
      Monitoring        Drift-stress macro-F1  0.9658              Performance under synthetic shift

CHEMGUARD MISSION CONTROL
Locked test batch: 54 samples
AUTO_ACCEPT: 48
MANUAL_REVIEW: 5
ANOMALY_REVIEW: 1
Automated-decision accuracy: 100.00%
Observed model errors sent to review: 1/1
PSI alert features in drift stress test: 6

✅ Confidence gate protected the automated path from the observed classification error


## 24. Recruiter demo — the wow factor

**ChemGuard AI** is no longer just a wine-classification exercise. It demonstrates an end-to-end **reliable AI workflow for laboratory quality control**:

- **98.15%** locked-test accuracy and **0.9829 macro-F1**
- **90.74%** coverage at the 0.80 confidence threshold with **100% selective accuracy**
- the **single observed classification error was routed away from automatic acceptance**
- independent anomaly screening for unusual samples
- measurement-noise stress testing
- synthetic distribution-shift testing and PSI monitoring
- explainability through permutation importance
- production-style schema validation and decision API logic
- an executive reliability scorecard and operational mission-control layer

**Interview story:** I did not stop at model accuracy. I designed the model to know when not to automate, created a human-review pathway, tested robustness under noise and drift, and packaged the outputs as operational decision intelligence.


## 25. Deployment Readiness Gate — can ChemGuard automate safely?

This final production layer converts model metrics into an explicit **go / guarded-go / no-go** decision. The thresholds are portfolio engineering targets, not regulatory standards. They force the system to prove balanced accuracy, calibration, safe automation coverage and robustness under synthetic drift before high-confidence samples are allowed onto the automated path.


In [20]:
deployment_checks = pd.DataFrame([
    ["Locked-test macro-F1 >= 0.95", TEST_MACRO_F1 >= 0.95, TEST_MACRO_F1],
    ["Expected Calibration Error <= 0.10", ECE <= 0.10, ECE],
    ["Auto coverage @ 0.80 >= 0.80", auto_at_80["auto_coverage"] >= 0.80, float(auto_at_80["auto_coverage"])],
    ["Selective accuracy @ 0.80 >= 0.99", auto_at_80["selective_accuracy"] >= 0.99, float(auto_at_80["selective_accuracy"])],
    ["Drift-stress macro-F1 >= 0.90", f1_score(y_test, drift_pred, average="macro") >= 0.90, float(f1_score(y_test, drift_pred, average="macro"))],
    ["Observed auto-path errors == 0", int((~batch_correct[auto_mask]).sum()) == 0, int((~batch_correct[auto_mask]).sum())],
], columns=["gate", "pass", "observed"])

print("CHEMGUARD DEPLOYMENT READINESS GATE")
print("=" * 84)
for _, row in deployment_checks.iterrows():
    status = "PASS" if row["pass"] else "FAIL"
    value = row["observed"]
    if isinstance(value, (float, np.floating)):
        value = f"{value:.4f}"
    print(f"{status} | {row['gate']} | observed={value}")

DEPLOYMENT_READY = bool(deployment_checks["pass"].all())
print("\nFINAL STATUS:", "GUARDED AUTOMATION READY ✅" if DEPLOYMENT_READY else "HUMAN-REVIEW ONLY ⚠️")


CHEMGUARD DEPLOYMENT READINESS GATE
PASS | Locked-test macro-F1 >= 0.95 | observed=0.9829
PASS | Expected Calibration Error <= 0.10 | observed=0.0560
PASS | Auto coverage @ 0.80 >= 0.80 | observed=0.9074
PASS | Selective accuracy @ 0.80 >= 0.99 | observed=1.0000
PASS | Drift-stress macro-F1 >= 0.90 | observed=0.9658
PASS | Observed auto-path errors == 0 | observed=0

FINAL STATUS: GUARDED AUTOMATION READY ✅


## 26. CV-ready impact statement

**ChemGuard AI — Reliable Laboratory ML System:** built an end-to-end chemical classification workflow with **98.15% locked-test accuracy**, **0.9829 macro-F1**, confidence-gated automation, anomaly screening, calibration analysis, measurement-noise testing, PSI drift monitoring and a deployment-readiness gate; the 0.80 confidence policy automated **90.74%** of held-out samples at **100% selective accuracy** in the benchmark while routing the observed classification error to review.

**Interview boundary:** the UCI Wine dataset is a small laboratory benchmark, so the project demonstrates the reliability-engineering pattern rather than claiming industrial or regulated deployment validation.
